In [2]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [3]:
df = pd.read_csv("crmls_last_6_months.csv", low_memory=False)

In [4]:
binary_vars = [
    'ViewYN', 'WaterfrontYN', 'BasementYN', 'PoolPrivateYN',
    'AttachedGarageYN', 'FireplaceYN', 'NewConstructionYN'
]

In [5]:
df.head()

,BuyerAgentAOR,ListAgentAOR,Flooring,ViewYN,WaterfrontYN,BasementYN,PoolPrivateYN,OriginalListPrice,ListingKey,ListAgentEmail,...,MainLevelBedrooms,NewConstructionYN,GarageSpaces,HighSchoolDistrict,PostalCode,AssociationFee,LotSizeSquareFeet,MiddleOrJuniorSchoolDistrict,latfilled,lonfilled
0,Glendale,Glendale,NaN,NaN,NaN,NaN,NaN,3350000.0,552665323,Listings@LockerRealty.com,...,NaN,False,0.0,NaN,91331,NaN,88445.0,NaN,NaN,NaN
1,InlandValleys,InlandValleys,NaN,True,NaN,NaN,NaN,125000.0,551977092,Patriciapandeyrealtor@gmail.com,...,NaN,False,NaN,NaN,92530,0.0,9148.0,NaN,NaN,NaN
2,PacificWest,PacificWest,NaN,True,NaN,NaN,NaN,300000.0,551927931,joshuacho1004@gmail.com,...,NaN,False,NaN,NaN,92356,0.0,6969600.0,NaN,NaN,NaN
3,Mlslistings,Mlslistings,"Carpet,Laminate,Tile",False,NaN,NaN,NaN,800000.0,544420694,assistant@danmoskowitz.com,...,NaN,False,0.0,Other,95122,NaN,6418.0,NaN,NaN,NaN
4,LakeCounty,LakeCounty,NaN,True,NaN,NaN,NaN,21500.0,516603298,jan@zapcom.net,...,NaN,False,NaN,NaN,95464,0.0,4559.0,NaN,NaN,NaN


In [6]:
features = ['NewConstructionYN', 'GarageSpaces', 'HighSchoolDistrict', 
            'PostalCode', 'LotSizeSquareFeet']

na_percent = df[features].isna().mean() * 100
print("Percentage of missing values in each feature:")
print(na_percent)

Percentage of missing values in each feature:
NewConstructionYN     11.899603
GarageSpaces          13.421803
HighSchoolDistrict    34.058721
PostalCode             0.023100
LotSizeSquareFeet      8.843256
dtype: float64


In [7]:
# 1. New Construction
# If NA means "not new construction" in context, replace NA with False. Cause if it is a new constuction, it should be actively posted online.
df['NewConstructionYN'] = df['NewConstructionYN'].fillna(False)

C:\Users\sarah\AppData\Local\Temp\ipykernel_17728\3581258723.py:3: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['NewConstructionYN'] = df['NewConstructionYN'].fillna(False)


In [8]:
# 2. Garage Space: NA probably means 0 garage spaces. Due to the same reason above.
# e.g. condo, older house 
df['GarageSpaces'] = df['GarageSpaces'].fillna(0)

In [9]:
df['PostalCode'] = df['PostalCode'].fillna('Unknown')

In [10]:
# check for negative values
for col in ['GarageSpaces', 'LotSizeSquareFeet']:
    neg_count = (df[col] < 0).sum()
    print(f"{col}: {neg_count} negative values")

GarageSpaces: 0 negative values
LotSizeSquareFeet: 0 negative values


In [11]:
# remove outliers
# check the percentage outside of 1.5 IQR
for col in ['GarageSpaces', 'LotSizeSquareFeet']:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    outliers = ((df[col] < lower) | (df[col] > upper)) & (~df[col].isna())
    outlier_pct = outliers.mean() * 100
    print(f"{col}: Would remove {outlier_pct:.2f}% as outliers using 1.5*IQR")

GarageSpaces: Would remove 0.44% as outliers using 1.5*IQR
LotSizeSquareFeet: Would remove 14.42% as outliers using 1.5*IQR


In [12]:
# good for garage spaces but not for lot size
# remove for garage 
col = 'GarageSpaces'
q1 = df[col].quantile(0.25)
q3 = df[col].quantile(0.75)
iqr = q3 - q1
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr
df = df[df[col].isna() | ((df[col] >= lower) & (df[col] <= upper))]

In [13]:
# check the upper bound
large_lots = (df['LotSizeSquareFeet'] > 20000).sum()
print(f"Number of homes with lot size > 20,000 sqft: {large_lots} ({large_lots / len(df) * 100:.2f}%)")

Number of homes with lot size > 20,000 sqft: 20850 (16.68%)


In [14]:
# For general California housing analysis, keep only homes with LotSizeSquareFeet ≤ 20,000 sqft
df_cleaned = df[df['LotSizeSquareFeet'].isna() | (df['LotSizeSquareFeet'] <= 20000)]

In [15]:
# final result
df_cleaned[features]

,NewConstructionYN,GarageSpaces,HighSchoolDistrict,PostalCode,LotSizeSquareFeet
1,False,0.0,NaN,92530,9148.0
3,False,0.0,Other,95122,6418.0
4,False,0.0,NaN,95464,4559.0
6,False,0.0,NaN,92354,5000.0
9,False,0.0,NaN,93446,8002.0
...,...,...,...,...,...
125532,False,0.0,NaN,94509,871.2
125533,False,0.0,NaN,95467,10179.0
125538,False,2.0,NaN,92078,NaN
125539,False,1.0,Encinitas Union,92024,1.0
